In [1]:
import os, sys, random, shutil, warnings, glob, time, itertools, math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
import torchvision.transforms as T
from torchvision.utils import save_image
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
from skimage.metrics import structural_similarity as ssim, peak_signal_noise_ratio as psnr
warnings.filterwarnings('ignore')

OUTPUT_DIR = '/kaggle/working'
CKPT_DIR = os.path.join(OUTPUT_DIR, 'checkpoints')
IMG_DIR = os.path.join(OUTPUT_DIR, 'images')
for d in [CKPT_DIR, IMG_DIR]:
    os.makedirs(d, exist_ok=True)

IMG_SIZE = 128
BATCH = 4
LR = 0.0002
BETAS = (0.5, 0.999)
EPOCHS = 50
LAMBDA_CYCLE = 10.0
LAMBDA_ID = 5.0
N_RES = 6
N_WORKERS = 2
MAX_SAMPLES = 2000

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = torch.cuda.is_available()
n_gpus = torch.cuda.device_count()
print(f'Device: {device} | GPUs: {n_gpus} | AMP: {USE_AMP}')

Device: cuda | GPUs: 2 | AMP: True


In [2]:
os.system('pip install -q datasets huggingface_hub gradio scikit-image')

0

In [3]:
from datasets import load_dataset
from huggingface_hub import snapshot_download

SKETCH_DIR = os.path.join(OUTPUT_DIR, 'sketches')
PHOTO_DIR = os.path.join(OUTPUT_DIR, 'photos')
os.makedirs(SKETCH_DIR, exist_ok=True)
os.makedirs(PHOTO_DIR, exist_ok=True)

def save_hf_sketches():
    try:
        ds = load_dataset('sdiaeyu6n/tu-berlin', split='train', streaming=True)
        count = 0
        for item in ds:
            if count >= MAX_SAMPLES:
                break
            img = item.get('image') or item.get('sketch') or (list(item.values())[0] if item else None)
            if img is None:
                continue
            if not isinstance(img, Image.Image):
                img = Image.fromarray(np.array(img))
            img = img.convert('RGB')
            img.save(os.path.join(SKETCH_DIR, f'tu_{count:05d}.jpg'))
            count += 1
        print(f'TU-Berlin sketches saved: {count}')
        return count
    except Exception as e:
        print(f'TU-Berlin load failed: {e}')
        return 0

def collect_sketchy():
    base = '/kaggle/input/sketchy-dataset'
    if not os.path.exists(base):
        print('Sketchy dataset not found at /kaggle/input/sketchy-dataset')
        return 0, 0
    sk_count = 0
    ph_count = 0
    for root, dirs, files in os.walk(base):
        for f in files:
            if not f.lower().endswith(('.jpg', '.jpeg', '.png')):
                continue
            src = os.path.join(root, f)
            rp = root.lower()
            if 'sketch' in rp and sk_count < MAX_SAMPLES:
                dst = os.path.join(SKETCH_DIR, f'sk_{sk_count:05d}.jpg')
                shutil.copy2(src, dst)
                sk_count += 1
            elif ('photo' in rp or 'image' in rp) and ph_count < MAX_SAMPLES:
                dst = os.path.join(PHOTO_DIR, f'ph_{ph_count:05d}.jpg')
                shutil.copy2(src, dst)
                ph_count += 1
    print(f'Sketchy: sketches={sk_count}, photos={ph_count}')
    return sk_count, ph_count

def collect_quickdraw():
    base = '/kaggle/input'
    count = 0
    for root, dirs, files in os.walk(base):
        if 'quickdraw' not in root.lower():
            continue
        for f in files:
            if not f.lower().endswith(('.jpg', '.jpeg', '.png')):
                continue
            if count >= MAX_SAMPLES:
                break
            src = os.path.join(root, f)
            dst = os.path.join(SKETCH_DIR, f'qd_{count:05d}.jpg')
            shutil.copy2(src, dst)
            count += 1
    print(f'QuickDraw sketches: {count}')
    return count

def generate_synthetic(n=500):
    os.makedirs(SKETCH_DIR, exist_ok=True)
    os.makedirs(PHOTO_DIR, exist_ok=True)
    for i in range(n):
        sk = np.ones((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8) * 255
        cx, cy = random.randint(20,108), random.randint(20,108)
        r = random.randint(10,40)
        for x in range(IMG_SIZE):
            for y in range(IMG_SIZE):
                if abs(math.sqrt((x-cx)**2+(y-cy)**2)-r) < 2:
                    sk[y,x] = [0,0,0]
        Image.fromarray(sk).save(os.path.join(SKETCH_DIR, f'syn_sk_{i:05d}.jpg'))
        ph = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
        color = [random.randint(50,200), random.randint(50,200), random.randint(50,200)]
        for x in range(IMG_SIZE):
            for y in range(IMG_SIZE):
                if math.sqrt((x-cx)**2+(y-cy)**2) < r:
                    ph[y,x] = color
        Image.fromarray(ph).save(os.path.join(PHOTO_DIR, f'syn_ph_{i:05d}.jpg'))
    print(f'Synthetic data generated: {n} pairs')

tu_count = save_hf_sketches()
sk_sketchy, ph_sketchy = collect_sketchy()
qd_count = collect_quickdraw()

total_sk = len(glob.glob(os.path.join(SKETCH_DIR, '*')))
total_ph = len(glob.glob(os.path.join(PHOTO_DIR, '*')))
print(f'Total sketches: {total_sk}, Total photos: {total_ph}')

if total_sk < 100 or total_ph < 100:
    print('Insufficient data, generating synthetic samples...')
    generate_synthetic(500)
    total_sk = len(glob.glob(os.path.join(SKETCH_DIR, '*')))
    total_ph = len(glob.glob(os.path.join(PHOTO_DIR, '*')))
    print(f'After synthetic: sketches={total_sk}, photos={total_ph}')

README.md: 0.00B [00:00, ?B/s]

TU-Berlin sketches saved: 2000
Sketchy dataset not found at /kaggle/input/sketchy-dataset
QuickDraw sketches: 0
Total sketches: 2000, Total photos: 0
Insufficient data, generating synthetic samples...
Synthetic data generated: 500 pairs
After synthetic: sketches=2500, photos=500


In [4]:
class DomainDataset(Dataset):
    def __init__(self, folder, transform=None, max_n=MAX_SAMPLES):
        exts = ('*.jpg', '*.jpeg', '*.png', '*.bmp')
        files = []
        for e in exts:
            files += glob.glob(os.path.join(folder, '**', e), recursive=True)
        random.shuffle(files)
        self.files = files[:max_n]
        self.transform = transform

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        try:
            img = Image.open(self.files[idx]).convert('RGB')
        except:
            img = Image.fromarray(np.ones((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8) * 128)
        if self.transform:
            img = self.transform(img)
        return img

tfm = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize([0.5]*3, [0.5]*3)
])

sk_ds = DomainDataset(SKETCH_DIR, tfm)
ph_ds = DomainDataset(PHOTO_DIR, tfm)
print(f'Sketch dataset: {len(sk_ds)}, Photo dataset: {len(ph_ds)}')

sk_loader = DataLoader(sk_ds, batch_size=BATCH, shuffle=True, num_workers=N_WORKERS, pin_memory=True, drop_last=True)
ph_loader = DataLoader(ph_ds, batch_size=BATCH, shuffle=True, num_workers=N_WORKERS, pin_memory=True, drop_last=True)
print(f'Sketch batches: {len(sk_loader)}, Photo batches: {len(ph_loader)}')

Sketch dataset: 2000, Photo dataset: 500
Sketch batches: 500, Photo batches: 125


In [5]:
class ResBlock(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.ReflectionPad2d(1),
            nn.Conv2d(ch, ch, 3),
            nn.InstanceNorm2d(ch),
            nn.ReLU(inplace=True),
            nn.ReflectionPad2d(1),
            nn.Conv2d(ch, ch, 3),
            nn.InstanceNorm2d(ch)
        )
    def forward(self, x):
        return x + self.block(x)

class Generator(nn.Module):
    def __init__(self, in_ch=3, out_ch=3, ngf=64, n_res=N_RES):
        super().__init__()
        layers = [
            nn.ReflectionPad2d(3),
            nn.Conv2d(in_ch, ngf, 7),
            nn.InstanceNorm2d(ngf),
            nn.ReLU(inplace=True)
        ]
        ch = ngf
        for _ in range(2):
            layers += [
                nn.Conv2d(ch, ch*2, 3, stride=2, padding=1),
                nn.InstanceNorm2d(ch*2),
                nn.ReLU(inplace=True)
            ]
            ch *= 2
        for _ in range(n_res):
            layers.append(ResBlock(ch))
        for _ in range(2):
            layers += [
                nn.ConvTranspose2d(ch, ch//2, 3, stride=2, padding=1, output_padding=1),
                nn.InstanceNorm2d(ch//2),
                nn.ReLU(inplace=True)
            ]
            ch //= 2
        layers += [
            nn.ReflectionPad2d(3),
            nn.Conv2d(ch, out_ch, 7),
            nn.Tanh()
        ]
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)

class PatchDiscriminator(nn.Module):
    def __init__(self, in_ch=3, ndf=64):
        super().__init__()
        def block(ic, oc, norm=True):
            layers = [nn.Conv2d(ic, oc, 4, stride=2, padding=1)]
            if norm:
                layers.append(nn.InstanceNorm2d(oc))
            layers.append(nn.LeakyReLU(0.2, inplace=True))
            return layers
        self.model = nn.Sequential(
            *block(in_ch, ndf, norm=False),
            *block(ndf, ndf*2),
            *block(ndf*2, ndf*4),
            nn.Conv2d(ndf*4, ndf*8, 4, padding=1),
            nn.InstanceNorm2d(ndf*8),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf*8, 1, 4, padding=1)
        )

    def forward(self, x):
        return self.model(x)

def init_weights(m):
    if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
        nn.init.normal_(m.weight, 0.0, 0.02)
        if m.bias is not None:
            nn.init.zeros_(m.bias)
    elif isinstance(m, nn.InstanceNorm2d) and m.weight is not None:
        nn.init.normal_(m.weight, 1.0, 0.02)
        nn.init.zeros_(m.bias)

G_AB = Generator().to(device)
G_BA = Generator().to(device)
D_A = PatchDiscriminator().to(device)
D_B = PatchDiscriminator().to(device)

for m in [G_AB, G_BA, D_A, D_B]:
    m.apply(init_weights)

if n_gpus > 1:
    G_AB = nn.DataParallel(G_AB)
    G_BA = nn.DataParallel(G_BA)
    D_A = nn.DataParallel(D_A)
    D_B = nn.DataParallel(D_B)

total_params = sum(p.numel() for p in G_AB.parameters())
print(f'Generator params: {total_params:,}')
print(f'Discriminator params: {sum(p.numel() for p in D_A.parameters()):,}')

Generator params: 7,837,699
Discriminator params: 2,764,737


In [6]:
criterion_adv = nn.MSELoss()
criterion_cycle = nn.L1Loss()
criterion_id = nn.L1Loss()

opt_G = torch.optim.Adam(
    itertools.chain(G_AB.parameters(), G_BA.parameters()),
    lr=LR, betas=BETAS
)
opt_DA = torch.optim.Adam(D_A.parameters(), lr=LR, betas=BETAS)
opt_DB = torch.optim.Adam(D_B.parameters(), lr=LR, betas=BETAS)

def lr_lambda(epoch):
    decay_start = EPOCHS // 2
    if epoch < decay_start:
        return 1.0
    return max(0.0, 1.0 - (epoch - decay_start) / decay_start)

sched_G = torch.optim.lr_scheduler.LambdaLR(opt_G, lr_lambda)
sched_DA = torch.optim.lr_scheduler.LambdaLR(opt_DA, lr_lambda)
sched_DB = torch.optim.lr_scheduler.LambdaLR(opt_DB, lr_lambda)

scaler = GradScaler() if USE_AMP else None

class ReplayBuffer:
    def __init__(self, max_size=50):
        self.buf = []
        self.max = max_size

    def push_pop(self, imgs):
        ret = []
        for img in imgs.data:
            img = img.unsqueeze(0)
            if len(self.buf) < self.max:
                self.buf.append(img)
                ret.append(img)
            elif random.random() > 0.5:
                i = random.randint(0, self.max - 1)
                ret.append(self.buf[i].clone())
                self.buf[i] = img
            else:
                ret.append(img)
        return torch.cat(ret, 0)

buf_A = ReplayBuffer()
buf_B = ReplayBuffer()
print('Optimizers, schedulers, replay buffers ready.')

Optimizers, schedulers, replay buffers ready.


In [7]:
hist = {'g': [], 'da': [], 'db': [], 'cycle': [], 'id': []}

def train_epoch(ep):
    G_AB.train(); G_BA.train(); D_A.train(); D_B.train()
    g_tot = da_tot = db_tot = cy_tot = id_tot = n = 0
    sk_it = iter(sk_loader)
    ph_it = iter(ph_loader)
    steps = min(len(sk_loader), len(ph_loader))

    for _ in range(steps):
        try:
            real_A = next(sk_it).to(device)
        except StopIteration:
            sk_it = iter(sk_loader)
            real_A = next(sk_it).to(device)
        try:
            real_B = next(ph_it).to(device)
        except StopIteration:
            ph_it = iter(ph_loader)
            real_B = next(ph_it).to(device)

        bs = real_A.size(0)
        ones = torch.ones(bs, 1, *D_A(real_A).shape[2:], device=device)
        zeros = torch.zeros_like(ones)

        opt_G.zero_grad()
        if USE_AMP:
            with autocast():
                fake_B = G_AB(real_A)
                fake_A = G_BA(real_B)
                rec_A = G_BA(fake_B)
                rec_B = G_AB(fake_A)
                id_A = G_BA(real_A)
                id_B = G_AB(real_B)
                pred_ones_shape = D_B(fake_B).shape
                ones_g = torch.ones(pred_ones_shape, device=device)
                l_adv = criterion_adv(D_B(fake_B), ones_g) + criterion_adv(D_A(fake_A), ones_g)
                l_cycle = (criterion_cycle(rec_A, real_A) + criterion_cycle(rec_B, real_B)) * LAMBDA_CYCLE
                l_id = (criterion_id(id_A, real_A) + criterion_id(id_B, real_B)) * LAMBDA_ID
                l_g = l_adv + l_cycle + l_id
            scaler.scale(l_g).backward()
            scaler.step(opt_G)
        else:
            fake_B = G_AB(real_A)
            fake_A = G_BA(real_B)
            rec_A = G_BA(fake_B)
            rec_B = G_AB(fake_A)
            id_A = G_BA(real_A)
            id_B = G_AB(real_B)
            pred_ones_shape = D_B(fake_B).shape
            ones_g = torch.ones(pred_ones_shape, device=device)
            l_adv = criterion_adv(D_B(fake_B), ones_g) + criterion_adv(D_A(fake_A), ones_g)
            l_cycle = (criterion_cycle(rec_A, real_A) + criterion_cycle(rec_B, real_B)) * LAMBDA_CYCLE
            l_id = (criterion_id(id_A, real_A) + criterion_id(id_B, real_B)) * LAMBDA_ID
            l_g = l_adv + l_cycle + l_id
            l_g.backward()
            opt_G.step()

        fake_B_buf = buf_B.push_pop(fake_B.detach())
        fake_A_buf = buf_A.push_pop(fake_A.detach())

        opt_DA.zero_grad()
        if USE_AMP:
            with autocast():
                da_shape = D_A(real_A).shape
                ones_d = torch.ones(da_shape, device=device)
                zeros_d = torch.zeros(da_shape, device=device)
                l_da = (criterion_adv(D_A(real_A), ones_d) + criterion_adv(D_A(fake_A_buf), zeros_d)) * 0.5
            scaler.scale(l_da).backward()
            scaler.step(opt_DA)
        else:
            da_shape = D_A(real_A).shape
            ones_d = torch.ones(da_shape, device=device)
            zeros_d = torch.zeros(da_shape, device=device)
            l_da = (criterion_adv(D_A(real_A), ones_d) + criterion_adv(D_A(fake_A_buf), zeros_d)) * 0.5
            l_da.backward()
            opt_DA.step()

        opt_DB.zero_grad()
        if USE_AMP:
            with autocast():
                db_shape = D_B(real_B).shape
                ones_db = torch.ones(db_shape, device=device)
                zeros_db = torch.zeros(db_shape, device=device)
                l_db = (criterion_adv(D_B(real_B), ones_db) + criterion_adv(D_B(fake_B_buf), zeros_db)) * 0.5
            scaler.scale(l_db).backward()
            scaler.step(opt_DB)
            scaler.update()
        else:
            db_shape = D_B(real_B).shape
            ones_db = torch.ones(db_shape, device=device)
            zeros_db = torch.zeros(db_shape, device=device)
            l_db = (criterion_adv(D_B(real_B), ones_db) + criterion_adv(D_B(fake_B_buf), zeros_db)) * 0.5
            l_db.backward()
            opt_DB.step()

        g_tot += l_g.item(); da_tot += l_da.item(); db_tot += l_db.item()
        cy_tot += l_cycle.item(); id_tot += l_id.item()
        n += 1

    return g_tot/n, da_tot/n, db_tot/n, cy_tot/n, id_tot/n

print('Training function defined.')

Training function defined.


In [ ]:
def save_samples(ep, sk_it, ph_it):
    G_AB.eval(); G_BA.eval()
    with torch.no_grad():
        try:
            real_A = next(sk_it).to(device)[:4]
        except:
            sk_it2 = iter(sk_loader)
            real_A = next(sk_it2).to(device)[:4]
        try:
            real_B = next(ph_it).to(device)[:4]
        except:
            ph_it2 = iter(ph_loader)
            real_B = next(ph_it2).to(device)[:4]
        fake_B = G_AB(real_A)
        fake_A = G_BA(real_B)
        rec_A = G_BA(fake_B)
        rec_B = G_AB(fake_A)
        imgs = torch.cat([real_A, fake_B, rec_A, real_B, fake_A, rec_B], 0)
        path = os.path.join(IMG_DIR, f'epoch_{ep:03d}.png')
        save_image(imgs * 0.5 + 0.5, path, nrow=4)
    G_AB.train(); G_BA.train()

print('Training loop starting...')
sk_sample_it = iter(sk_loader)
ph_sample_it = iter(ph_loader)

for ep in range(1, EPOCHS + 1):
    t0 = time.time()
    lg, lda, ldb, lcy, lid = train_epoch(ep)
    hist['g'].append(lg); hist['da'].append(lda); hist['db'].append(ldb)
    hist['cycle'].append(lcy); hist['id'].append(lid)
    sched_G.step(); sched_DA.step(); sched_DB.step()
    elapsed = time.time() - t0
    print(f'Ep {ep:03d}/{EPOCHS} | G:{lg:.4f} DA:{lda:.4f} DB:{ldb:.4f} Cyc:{lcy:.4f} Id:{lid:.4f} | {elapsed:.1f}s')
    if ep % 5 == 0:
        save_samples(ep, sk_sample_it, ph_sample_it)
        state = {
            'epoch': ep,
            'G_AB': G_AB.state_dict(),
            'G_BA': G_BA.state_dict(),
            'D_A': D_A.state_dict(),
            'D_B': D_B.state_dict(),
            'opt_G': opt_G.state_dict(),
            'opt_DA': opt_DA.state_dict(),
            'opt_DB': opt_DB.state_dict(),
            'hist': hist
        }
        torch.save(state, os.path.join(CKPT_DIR, f'ckpt_ep{ep:03d}.pth'))
        print(f'  Checkpoint saved.')

torch.save(state, os.path.join(OUTPUT_DIR, 'cyclegan_final.pth'))
torch.save(G_AB.state_dict(), os.path.join(OUTPUT_DIR, 'G_AB_final.pth'))
torch.save(G_BA.state_dict(), os.path.join(OUTPUT_DIR, 'G_BA_final.pth'))
print('Training complete. Final models saved.')

Training loop starting...
Ep 001/50 | G:4.9325 DA:0.5093 DB:0.5750 Cyc:2.2751 Id:1.1306 | 61.4s
Ep 002/50 | G:2.4018 DA:0.2359 DB:0.2478 Cyc:0.9873 Id:0.4468 | 59.4s
Ep 003/50 | G:2.2944 DA:0.2073 DB:0.2195 Cyc:0.9560 Id:0.3985 | 60.3s
Ep 004/50 | G:2.2810 DA:0.1888 DB:0.2298 Cyc:0.9586 Id:0.3832 | 60.5s
Ep 005/50 | G:2.5188 DA:0.1619 DB:0.2086 Cyc:1.0403 Id:0.3985 | 59.4s
  Checkpoint saved.
Ep 006/50 | G:2.3826 DA:0.1633 DB:0.2023 Cyc:0.9571 Id:0.3762 | 60.4s
Ep 007/50 | G:2.4004 DA:0.1581 DB:0.2028 Cyc:0.9362 Id:0.3524 | 60.8s
Ep 008/50 | G:2.4205 DA:0.1328 DB:0.2021 Cyc:0.9256 Id:0.3447 | 60.4s
Ep 009/50 | G:2.4231 DA:0.1728 DB:0.1946 Cyc:0.9676 Id:0.3476 | 60.2s
Ep 010/50 | G:2.5035 DA:0.1311 DB:0.1861 Cyc:1.0212 Id:0.3584 | 61.3s
  Checkpoint saved.
Ep 011/50 | G:2.4374 DA:0.1175 DB:0.1733 Cyc:0.9363 Id:0.3388 | 61.2s
Ep 012/50 | G:2.3169 DA:0.1169 DB:0.2004 Cyc:0.9040 Id:0.3243 | 61.3s
Ep 013/50 | G:2.5394 DA:0.0980 DB:0.1750 Cyc:0.9558 Id:0.3279 | 60.7s
Ep 014/50 | G:2.4815 DA:

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
eps = range(1, EPOCHS + 1)
axes[0].plot(eps, hist['g'], 'b-', linewidth=1.5, label='Generator')
axes[0].set_title('Generator Loss vs Epochs', fontsize=14)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(eps, hist['da'], 'r-', linewidth=1.5, label='D_A (Sketch)')
axes[1].plot(eps, hist['db'], 'g-', linewidth=1.5, label='D_B (Photo)')
axes[1].set_title('Discriminator Loss vs Epochs', fontsize=14)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

axes[2].plot(eps, hist['cycle'], 'm-', linewidth=1.5, label='Cycle')
axes[2].plot(eps, hist['id'], 'c-', linewidth=1.5, label='Identity')
axes[2].set_title('Cycle & Identity Loss vs Epochs', fontsize=14)
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Loss')
axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Training curves saved.')

In [ ]:
def tensor_to_np(t):
    img = t.detach().cpu().squeeze(0)
    img = img * 0.5 + 0.5
    img = img.clamp(0, 1).permute(1, 2, 0).numpy()
    return (img * 255).astype(np.uint8)

def compute_metrics(real, fake):
    r = tensor_to_np(real).astype(np.float64) / 255.0
    f = tensor_to_np(fake).astype(np.float64) / 255.0
    s = ssim(r, f, multichannel=True, data_range=1.0, channel_axis=2)
    p = psnr(r, f, data_range=1.0)
    return s, p

G_AB.eval(); G_BA.eval()
ssim_ab_list, psnr_ab_list = [], []
ssim_ba_list, psnr_ba_list = [], []

with torch.no_grad():
    for i, (batch_A, batch_B) in enumerate(zip(sk_loader, ph_loader)):
        if i >= 10:
            break
        real_A = batch_A.to(device)
        real_B = batch_B.to(device)
        fake_B = G_AB(real_A)
        fake_A = G_BA(real_B)
        rec_A = G_BA(fake_B)
        rec_B = G_AB(fake_A)
        for j in range(real_A.size(0)):
            s, p = compute_metrics(real_A[j], rec_A[j])
            ssim_ab_list.append(s); psnr_ab_list.append(p)
            s, p = compute_metrics(real_B[j], rec_B[j])
            ssim_ba_list.append(s); psnr_ba_list.append(p)

print(f'Sketch Cycle SSIM: {np.mean(ssim_ab_list):.4f} | PSNR: {np.mean(psnr_ab_list):.2f} dB')
print(f'Photo  Cycle SSIM: {np.mean(ssim_ba_list):.4f} | PSNR: {np.mean(psnr_ba_list):.2f} dB')

metrics = {
    'sketch_cycle_ssim': float(np.mean(ssim_ab_list)),
    'sketch_cycle_psnr': float(np.mean(psnr_ab_list)),
    'photo_cycle_ssim': float(np.mean(ssim_ba_list)),
    'photo_cycle_psnr': float(np.mean(psnr_ba_list))
}
import json
with open(os.path.join(OUTPUT_DIR, 'metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=2)
print('Metrics saved.')

In [ ]:
tfm_eval = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize([0.5]*3, [0.5]*3)
])

G_AB.eval(); G_BA.eval()
fig, axes = plt.subplots(5, 6, figsize=(20, 16))
cols = ['Input Sketch', 'Sketch->Photo', 'Reconstructed Sketch', 'Input Photo', 'Photo->Sketch', 'Reconstructed Photo']
for c, ax in zip(cols, axes[0]):
    ax.set_title(c, fontsize=9, fontweight='bold')

sk_files = glob.glob(os.path.join(SKETCH_DIR, '*.jpg'))[:5]
ph_files = glob.glob(os.path.join(PHOTO_DIR, '*.jpg'))[:5]

if len(sk_files) < 5:
    sk_files = sk_files + [sk_files[0]] * (5 - len(sk_files))
if len(ph_files) < 5:
    ph_files = ph_files + [ph_files[0]] * (5 - len(ph_files))

with torch.no_grad():
    for row in range(5):
        real_A = tfm_eval(Image.open(sk_files[row]).convert('RGB')).unsqueeze(0).to(device)
        real_B = tfm_eval(Image.open(ph_files[row]).convert('RGB')).unsqueeze(0).to(device)
        fake_B = G_AB(real_A)
        rec_A = G_BA(fake_B)
        fake_A = G_BA(real_B)
        rec_B = G_AB(fake_A)
        imgs_row = [real_A, fake_B, rec_A, real_B, fake_A, rec_B]
        for col, img_t in enumerate(imgs_row):
            axes[row, col].imshow(tensor_to_np(img_t))
            axes[row, col].axis('off')

plt.suptitle('CycleGAN: Sketch <-> Photo Translation', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'qualitative_results.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Qualitative visualization saved.')

In [ ]:
import gradio as gr

G_AB_inf = Generator().to(device)
G_BA_inf = Generator().to(device)

g_ab_sd = G_AB.module.state_dict() if isinstance(G_AB, nn.DataParallel) else G_AB.state_dict()
g_ba_sd = G_BA.module.state_dict() if isinstance(G_BA, nn.DataParallel) else G_BA.state_dict()
G_AB_inf.load_state_dict(g_ab_sd)
G_BA_inf.load_state_dict(g_ba_sd)
G_AB_inf.eval(); G_BA_inf.eval()

def translate(image, direction):
    if image is None:
        return None
    if isinstance(image, np.ndarray):
        image = Image.fromarray(image.astype(np.uint8)).convert('RGB')
    img_t = tfm_eval(image).unsqueeze(0).to(device)
    with torch.no_grad():
        if direction == 'Sketch -> Photo':
            out = G_AB_inf(img_t)
        else:
            out = G_BA_inf(img_t)
    result = tensor_to_np(out)
    return result

with gr.Blocks(title='CycleGAN Sketch-Photo Translation') as demo:
    gr.Markdown('## CycleGAN: Sketch Photo Domain Translation')
    gr.Markdown('Upload an image and select translation direction.')
    with gr.Row():
        with gr.Column():
            inp_img = gr.Image(label='Input Image', type='numpy')
            direction = gr.Radio(
                choices=['Sketch -> Photo', 'Photo -> Sketch'],
                value='Sketch -> Photo',
                label='Translation Direction'
            )
            btn = gr.Button('Translate', variant='primary')
        with gr.Column():
            out_img = gr.Image(label='Translated Output')
    btn.click(fn=translate, inputs=[inp_img, direction], outputs=out_img)
    gr.Examples(
        examples=[[sk_files[0], 'Sketch -> Photo'], [ph_files[0], 'Photo -> Sketch']] if sk_files and ph_files else [],
        inputs=[inp_img, direction]
    )

demo.launch(share=True, quiet=True)
print('Gradio app launched.')

In [ ]:
print('Output directory contents:')
for f in sorted(os.listdir(OUTPUT_DIR)):
    fp = os.path.join(OUTPUT_DIR, f)
    if os.path.isfile(fp):
        size_mb = os.path.getsize(fp) / 1e6
        print(f'  {f}: {size_mb:.2f} MB')
    else:
        n_files = len(os.listdir(fp))
        print(f'  {f}/ ({n_files} files)')

print(f'\nFinal Metrics Summary:')
print(f'  Sketch Cycle SSIM : {metrics["sketch_cycle_ssim"]:.4f}')
print(f'  Sketch Cycle PSNR : {metrics["sketch_cycle_psnr"]:.2f} dB')
print(f'  Photo  Cycle SSIM : {metrics["photo_cycle_ssim"]:.4f}')
print(f'  Photo  Cycle PSNR : {metrics["photo_cycle_psnr"]:.2f} dB')